In [10]:
import requests
import pandas as pd
import osmnx as ox
import os
from dotenv import load_dotenv

load_dotenv()
TDX_CLIENT_ID = os.getenv("TDX_CLIENT_ID")
TDX_CLIENT_SECRET = os.getenv("TDX_CLIENT_SECRET")

In [13]:
def get_token():
    auth_url = "https://tdx.transportdata.tw/auth/realms/TDXConnect/protocol/openid-connect/token"
    auth_data = {
        'grant_type': 'client_credentials',
        'client_id': TDX_CLIENT_ID,
        'client_secret': TDX_CLIENT_SECRET
    }
    res = requests.post(auth_url, data=auth_data)
    return res.json().get('access_token')

In [14]:
def check_garbage_trucks():
    token = get_token()
    if not token:
        print("❌ 無法取得 Token，請檢查 .env 設定")
        return

    # 台南垃圾車即時位置 API
    url = "https://tdx.transportdata.tw/api/basic/v2/Env/GarbageTruck/Location/City/Tainan"
    headers = {"Authorization": f"Bearer {token}"}
    
    print(f"🚀 正在連線至 TDX 抓取台南垃圾車資訊...")
    response = requests.get(url, headers=headers)
    
    if response.status_code == 200:
        trucks = response.json()
        print(f"✅ 成功連線！目前全台南市共有 {len(trucks)} 台垃圾車在線上。")
        print("-" * 50)
        
        # 只印出前 10 台看看
        for truck in trucks[:10]:
            plate = truck.get("CarBusNo", "未知車牌")
            lat = truck.get("Latitude")
            lon = truck.get("Longitude")
            # 判斷是否有移動 (0 代表靜止，1 代表移動中)
            status = "移動中" if truck.get("IsMoving") == 1 else "靜止"
            
            print(f"🚛 車牌: {plate} | 狀態: {status}")
            print(f"📍 座標: {lat}, {lon}")
            print(f"🔗 Google Maps 連結: https://www.google.com/maps?q={lat},{lon}")
            print("-" * 30)
            
        if len(trucks) == 0:
            print("⚠️ 目前沒有垃圾車正在出勤（可能非收運時間）。")
    else:
        print(f"❌ API 請求失敗，錯誤碼: {response.status_code}")

if __name__ == "__main__":
    check_garbage_trucks()

🚀 正在連線至 TDX 抓取台南垃圾車資訊...
❌ API 請求失敗，錯誤碼: 404


In [ ]:


def apply_garbage_risk(G, client_id, client_secret):
    """取得台南垃圾車位置並注入路網權重"""
    token = get_tdx_token(client_id, client_secret)
    if not token:
        return G

    # TDX 台南垃圾車即時位置 API
    url = "https://tdx.transportdata.tw/api/basic/v2/Env/GarbageTruck/Location/City/Tainan"
    headers = {"Authorization": f"Bearer {token}"}
    
    try:
        response = requests.get(url, headers=headers)
        if response.status_code == 200:
            trucks = response.json()
            
            for truck in trucks:
                lat = truck.get("Latitude")
                lon = truck.get("Longitude")
                plate = truck.get("CarBusNo", "Unknown") # 車牌，可以顯示在 Popup
                
                if lat and lon:
                    # 找到離垃圾車最近的邊 (Edge)
                    # 我們假設垃圾車會堵住一整條路，所以用 nearest_edges
                    try:
                        u, v, k = ox.distance.nearest_edges(G, X=lon, Y=lat)
                        
                        # 🔴 核心避險邏輯：
                        # 垃圾車停靠通常很久，我們把這條路的「成本」加重 1000 公尺
                        # 這會逼演算法「除非繞路要多走一公里，否則請繞開這條路」
                        G[u][v][k]['dynamic_cost'] = G[u][v][k].get('dynamic_cost', G[u][v][k]['length']) + 1000
                        
                        # 標註這條邊已經被垃圾車佔據 (視覺化備用)
                        G[u][v][k]['has_garbage_truck'] = True
                        G[u][v][k]['truck_plate'] = plate
                    except:
                        continue
        return G
    except Exception as e:
        print(f"垃圾車 API 抓取失敗: {e}")
        return G